# Biblical Qwen3.6 27B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** `unsloth/Qwen3.6-27B` — bf16 weights (55.6 GB, `Qwen3_5ForConditionalGeneration`), quantized to bnb 4-bit on the fly by `load_in_4bit=True`

**Dataset:** Per-persona datagen JSONL — each persona has its own system prompt and distinctive voice

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Chat Template:** Tokenizer-native Qwen ChatML template (`<|im_start|>role\ncontent<|im_end|>`), applied via `tokenizer.apply_chat_template`

**Thinking mode:** Qwen3.6 is a thinking model. `ENABLE_THINKING` in the config cell toggles it for the inference/eval cells. See the comment there for exactly what it changes.

This notebook is modeled on `biblical_gemma4_12b_unsloth_4bit.ipynb`, which produced working LoRA adapters on this machine. Its DPO counterpart is `biblical_qwen3_6_27b_dpo_unsloth_4bit.ipynb`.

## 1. Configuration

In [1]:
import os

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = "/workspace/training/biblical"
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = "/workspace/biblical"
else:
    PROJECT_ROOT = "/home/spark/projects/training/biblical"

OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== SHARED HUGGING FACE CACHE ===========================
# The bf16 base is 55.6 GB. Point the Hub cache at the cache shared with vLLM so it
# is downloaded once and reused. Must be set BEFORE unsloth/transformers are imported.
for _cache_dir in ("/root/.cache/huggingface", "/home/spark/.cache/huggingface"):
    if os.path.isdir(_cache_dir):
        os.environ["HF_HUB_CACHE"] = _cache_dir
        break

# =========================== MODEL CONFIGURATION ===========================
# unsloth/Qwen3.6-27B ships bf16 weights (55.6 GB, Qwen3_5ForConditionalGeneration,
# model_type qwen3_5). There is no Unsloth pre-quantized bnb-4bit repo for Qwen3.6-27B,
# so Unsloth quantizes to bnb 4-bit on the fly via load_in_4bit=True below — the same
# pattern used by biblical_gemma4_12b_unsloth_4bit.ipynb.
BASE_LLM = "unsloth/Qwen3.6-27B"
MODEL_NAME_BASE = "biblical_qwen3_6_27b_unsloth_4bit"

# =========================== THINKING MODE ===========================
# Qwen3.6's chat template accepts `enable_thinking`. It affects ONLY the generation
# prompt (i.e. calls with add_generation_prompt=True):
#   False -> '<|im_start|>assistant\n<think>\n\n</think>\n\n'  (empty block; answer directly)
#   True  -> '<|im_start|>assistant\n<think>\n'                 (model opens a reasoning block)
# The SFT training text is rendered with add_generation_prompt=False, so this flag does
# NOT change what the model is trained on — it only controls the inference/eval cells.
# Note: the training data contains plain answers with no <think> content, so a model
# tuned here has not been taught to fill a reasoning block with persona-consistent text.
ENABLE_THINKING = False

# =========================== INPUT DATA ===========================
# Combined multi-turn ShareGPT JSONL from the datagen notebooks (per-persona + augmented).
# Already quality-filtered, multi-turn (4 QA pairs per conversation), grouped by topic.
INPUT_DATA_FILE = f"{PROJECT_ROOT}/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl"

# =========================== PERSONA SYSTEM PROMPTS ===========================
# System prompts are EXTRACTED from the JSONL at load time (see the data loading cell).
# This keeps training in sync with datagen — regenerate data with new/changed prompts and
# this notebook picks them up automatically. After loading, `persona_system_prompts` maps
# persona_key -> full prompt text, and is saved alongside the LoRA adapters for inference.

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
# MAX_SEQ_LENGTH sized from this exact dataset, measured with the Qwen3.6 tokenizer:
# 4352 examples, min 497 / mean 1463 / p99 2492 / max 2996 tokens. Nothing exceeds
# 3072, so this truncates nothing while cutting the padded worst case 25% vs 4096.
MAX_SEQ_LENGTH = 3072
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
TARGET_EPOCHS = 1

# =========================== CHECKPOINTING ===========================
# One epoch is ~544 steps and runs for hours. Checkpoint often enough that a crash
# costs minutes instead of a day; the training cell auto-resumes from the newest.
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3

# gc.collect() + torch.cuda.empty_cache() cadence during training. Aligned with the
# checkpoint interval - right after a save is when transient buffers are safe to drop.
CLEANUP_STEPS = 50

# =========================== LoRA CONFIGURATION ===========================
# Adapter is merged into the base weights at GGUF export, so adapter size on disk is
# irrelevant. Full attention + MLP targets and a higher rank for stronger persona learning.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
# Qwen sampling defaults used across the other Qwen notebooks in this project.
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.8
GEN_TOP_K = 20

# ============================================================================
print("Configuration loaded (Qwen3.6 27B 4-bit QLoRA)")
print(f"  Project root:    {PROJECT_ROOT}")
print(f"  HF hub cache:    {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:      {BASE_LLM}")
print(f"  Model name:      {MODEL_NAME_BASE}")
print(f"  Input data:      {INPUT_DATA_FILE}")
print(f"  Output base:     {OUTPUT_BASE_DIR}")
print(f"  LoRA output:     {LORA_OUTPUT_DIR}")
print(f"  LoRA config:     r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training:        batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Max seq length:  {MAX_SEQ_LENGTH} (dataset max is 2996 tokens)")
print(f"  Checkpoints:     every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Thinking mode:   {'ON' if ENABLE_THINKING else 'OFF'} (inference/eval cells only)")
print(f"  Persona prompts: extracted from JSONL at load time")

if not os.path.exists(INPUT_DATA_FILE):
    raise FileNotFoundError(f"Training data not found: {INPUT_DATA_FILE}")

Configuration loaded (Qwen3.6 27B 4-bit QLoRA)
  Project root:    /workspace/training/biblical
  HF hub cache:    /root/.cache/huggingface
  Base model:      unsloth/Qwen3.6-27B
  Model name:      biblical_qwen3_6_27b_unsloth_4bit
  Input data:      /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  Output base:     /workspace/training/biblical/output/biblical_qwen3_6_27b_unsloth_4bit
  LoRA output:     /workspace/training/biblical/output/biblical_qwen3_6_27b_unsloth_4bit/lora_adapters
  LoRA config:     r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Training:        batch=2, grad_accum=4, lr=0.0002
  Max seq length:  3072 (dataset max is 2996 tokens)
  Checkpoints:     every 50 steps, keep 3
  Thinking mode:   OFF (inference/eval cells only)
  Persona prompts: extracted from JSONL at load time


## 2. Environment Preparation

Safe to re-run. Order matters:

1. Verify the CUDA PyTorch build is intact (no side effects).
2. Core training packages, and remove the aarch64 `torchao` build that blocks PEFT's bnb 4-bit dispatcher.
3. `transformers` + `peft` from git main — the `qwen3_5` architecture is newer than the container's release.
4. Small utility packages.
5. Rebuild `causal_conv1d` from source — the NGC image ships the Python package **without** its compiled CUDA extension, which hard-crashes any import that touches the FalconH1 model inside `transformers`/`unsloth`.
6. `flash-linear-attention` — Triton kernels for the Qwen3.6 gated-delta-rule fast path.
7. Import `unsloth` **before** `transformers` so its monkey-patches apply.

**Restart the kernel after this cell, then resume from Cell 1.**

In [2]:
import os, sys, subprocess, importlib, importlib.util

def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall - peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
_pip("uninstall", "-y", "-q", "torchao")

# --- 3. transformers + peft from git main -------------------------------------
# Qwen3.6 uses the `qwen3_5` architecture, which is newer than the transformers
# release the container ships. Install from git main so the arch is recognised.
print("  Installing transformers + peft from git main...")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image ships the causal_conv1d Python package WITHOUT its compiled CUDA
# extension (causal_conv1d_cuda). That hard-crashes any import that reaches the
# FalconH1 model inside transformers or unsloth, so it must be fixed BEFORE
# importing either.
#
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required to
# force a source build, together with CAUSAL_CONV1D_FORCE_BUILD=TRUE. The first
# build takes a few minutes on aarch64; pip caches the result afterwards.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up - don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. flash-linear-attention ------------------------------------------------
# fla provides the Triton JIT kernels (chunk_gated_delta_rule etc.) used by the
# Qwen3.6 fast path. fla-core installs into the same `fla` namespace.
if _check_import("fla") is None:
    print("  Installing flash-linear-attention...")
    _pip("install", "-q", "--no-deps", "flash-linear-attention", "fla-core")

_fast_path_ok = False
try:
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
    _fast_path_ok = _causal_ok and chunk_gated_delta_rule is not None
    for _k in list(sys.modules.keys()):
        if _k.startswith("fla."):
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError):
    pass
print(f"  Fast path: {'ENABLED' if _fast_path_ok else 'DISABLED (using torch fallback)'}")

# --- 7. Import unsloth FIRST, then transformers -------------------------------
# Unsloth must be imported before transformers/trl/peft so its monkey-patches apply.
# Purge anything already loaded so the imports pick up the versions installed above.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  fla:           {'OK' if _check_import('fla') else 'MISSING'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
print()
print("Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 - CUDA 13.0 - GPU: NVIDIA GB10
  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...
  Installing transformers + peft from git main...
  causal_conv1d: OK (CUDA extension loaded)
  Fast path: ENABLED
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  unsloth:       2026.7.5
  transformers:  5.15.0.dev0
  peft:          0.20.0
  trl:           0.24.0
  causal_conv1d: OK
  fla:           OK
  torchao:       absent (correct)

Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).


## 3. Load Dataset

Load the combined multi-turn ShareGPT JSONL from the datagen notebooks.

- Already quality-filtered (no short answers, no AI refusals)
- Multi-turn: 4 QA pairs grouped per conversation by topic
- Each conversation has a persona-specific system prompt
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`

In [3]:
import json, os, re
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING COMBINED SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

# Load multi-turn conversations and EXTRACT system prompts from the JSONL.
# This replaces hardcoded prompt dicts — prompts stay in sync with datagen automatically.
conversations = []
persona_system_prompts = {}   # persona_key -> full system prompt text
persona_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)

        # Extract persona name from "You are <Name>, ..." pattern
        sys_msg = conv["conversations"][0]["value"]
        match = re.match(r"You are (.+?),", sys_msg)
        if match:
            raw_name = match.group(1)
            # Normalize to snake_case key: lowercase, strip leading "the ", underscores for spaces
            key = raw_name.lower()
            key = re.sub(r"^the\s+", "", key)
            key = key.replace(" ", "_")
            persona_counts[key] += 1
            if key not in persona_system_prompts:
                persona_system_prompts[key] = sys_msg
        else:
            print(f"  WARNING: could not extract persona from system prompt: {sys_msg[:80]}...")

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} multi-turn conversations across {len(persona_counts)} personas")
print(f"Extracted {len(persona_system_prompts)} unique system prompts from JSONL")
print(f"Columns: {dataset.column_names}")
print(f"\nPer-persona breakdown:")
for p, c in sorted(persona_counts.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d} conversations")

# Show a sample prompt to verify extraction
sample_key = next(iter(persona_system_prompts))
print(f"\n--- Sample extracted prompt ({sample_key}, first 200 chars) ---")
print(f"  {persona_system_prompts[sample_key][:200]}...")

LOADING COMBINED SHAREGPT DATA
  File: /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl

Total dataset: 4352 multi-turn conversations across 26 personas
Extracted 26 unique system prompts from JSONL
Columns: ['conversations', 'data_type']

Per-persona breakdown:
  moses                  785 conversations
  jeremiah               381 conversations
  paul                   377 conversations
  david                  350 conversations
  ezekiel                341 conversations
  isaiah                 332 conversations
  solomon                254 conversations
  job                    244 conversations
  daniel                 239 conversations
  peter                  148 conversations
  zechariah              145 conversations
  hosea                  107 conversations
  amos                    92 conversations
  joshua                  88 conversations
  micah                   73 conversations
  apostle_john            71 co

## 4. Validate & Summarize Dataset

Datagen data is already clean (no artifacts to strip). Verify data quality and show persona distribution.

In [4]:
bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count >=3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (should match extracted count: {len(persona_system_prompts)})")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\nBad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\nFiltering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Persona distribution
print(f"\nPERSONA DISTRIBUTION:")
max_name_len = max(len(n) for n in persona_counts)
for name, count in sorted(persona_counts.items(), key=lambda x: -x[1]):
    bar = "#" * (count // 50)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(persona_counts.values()):>5}")

# Show voice differentiation — first response from different personas
print(f"\nVOICE SAMPLES (first ~100 chars of response):")
seen_personas = set()
for example in dataset:
    system = example["conversations"][0]["value"]
    name_part = system.split(",")[0].replace("You are ", "")
    if name_part not in seen_personas and len(seen_personas) < 4:
        response_start = example["conversations"][2]["value"][:100]
        print(f"  {name_part}: \"{response_start}...\"")
        seen_personas.add(name_part)

print(f"\nDataset validated and ready for training")

DATA QUALITY CHECK
  Total examples: 4352
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 26 (should match extracted count: 26)
  Turn distribution: {3: 1741, 5: 91, 7: 541, 9: 1979}

PERSONA DISTRIBUTION:
  moses          785  ###############
  jeremiah       381  #######
  paul           377  #######
  david          350  #######
  ezekiel        341  ######
  isaiah         332  ######
  solomon        254  #####
  job            244  ####
  daniel         239  ####
  peter          148  ##
  zechariah      145  ##
  hosea          107  ##
  amos            92  #
  joshua          88  #
  micah           73  #
  apostle_john    71  #
  james           54  #
  malachi         44  
  joel            44  
  zephaniah       37  
  habakkuk        32  
  jonah           29  
  nahum           27  
  haggai          24  
  obadiah         19  
  jude            15  
  TOTAL         4352

VOICE SAMPLES (first ~100 chars of response):
  Daniel: "Four is the number that stay

## 5. Load Model & Tokenizer (4-bit)

Loads `unsloth/Qwen3.6-27B` bf16 weights and quantizes to bnb NF4 on the fly.

**First run downloads 55.6 GB** into the shared HF cache configured in Cell 1. Subsequent runs reuse it.

In [5]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the broken
# torch.compile path. Carried over from the working Gemma 4 12B notebook on this machine.
# Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Qwen3.6 is Qwen3_5ForConditionalGeneration, so Unsloth can return a Processor
# object rather than a plain tokenizer.
if hasattr(tokenizer, "vocab_size"):
    vocab_size = tokenizer.vocab_size
elif hasattr(tokenizer, "tokenizer") and hasattr(tokenizer.tokenizer, "vocab_size"):
    vocab_size = tokenizer.tokenizer.vocab_size
else:
    vocab_size = "unknown"

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit (QLoRA, quantized on load)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {vocab_size}")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.15.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Model loaded: unsloth/Qwen3.6-27B
  Precision: 4-bit (QLoRA, quantized on load)
  Max sequence length: 3072
  Vocab size: 248044
  Attn impl: flash_attention_2
  GPU allocated: 18.0 GB


## 6. Format Dataset for Chat Template

Apply Qwen's ChatML template to each conversation and build the final training dataset.

Rendered with `add_generation_prompt=False`, so `ENABLE_THINKING` does not affect this step — full assistant turns come straight from the dataset.

In [6]:
# Standardize ShareGPT rows, then format with the Qwen tokenizer chat template
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset
dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
dataset = dataset.shuffle(seed=42)

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]["text"][:500])
print(f"\nDataset formatted: {len(dataset)} examples")

Unsloth: Standardizing formats (num_proc=24):   0%|          | 0/4352 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4352 [00:00<?, ? examples/s]

--- Sample formatted text (first 500 chars) ---
<|im_start|>system
You are David, the king of Israel, once a shepherd boy and warrior, the sweet psalmist who poured out your heart in song, a man after God's own heart who knew both triumph and deep sin.

YOUR DISTINCTIVE VOICE: Poetic, emotional, lyrical. Psalm cadence — parallelism, repetition, selah-like pauses. Shifts between ecstatic praise, raw anguish, and intimate confession. Uses nature imagery (shepherd, waters, mountains). Addresses God directly ('O LORD'). Unashamed emotional vulner

Dataset formatted: 4352 examples


## 7. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See Cell 1 config for module options.

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # Deliberately True, NOT "unsloth". The "unsloth" path copies saved activations into
    # pinned CPU buffers that only ever grow - unsloth_zoo/gradient_checkpointing.py
    # resizes them up and never down. On GB10 host and device share one 128 GB pool, so
    # that offload frees no capacity while ratcheting unreclaimable pinned pages upward
    # for hours. True uses standard recompute checkpointing with no host copies.
    use_gradient_checkpointing=True,
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES})")
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters added (r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
Trainable parameters: 159,383,552 / 15,135,298,800 (1.05%)


## 8. Trainer Setup

- 1 epoch — gently teaches persona-switching without degrading base capabilities
- Each example has a persona-specific system prompt so the model learns distinct voices

In [8]:
import gc
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead Python objects and return cached CUDA blocks to the allocator.

    Runs every `every` optimizer steps. This does not repair a leak - it releases
    cached/fragmented blocks that would otherwise sit reserved but unused, which
    matters on GB10 where host and device draw from the same 128 GB pool.
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
        return control


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        # Explicitly False. Qwen3.6 is a hybrid linear-attention model; its gated-delta
        # recurrent state and causal conv1d leak across sequence boundaries once packing
        # flattens a batch, so Unsloth force-disables packing for it regardless. The
        # previous run printed "Sample packing skipped (processor-based model detected)"
        # - packing=True was a silent lie about what actually ran.
        packing=False,
        # NOTE: train_sampling_strategy="group_by_length" does NOT work with this model
        # and is deliberately absent. Two independent blockers, both verified in
        # transformers source:
        #   1. Trainer._get_train_sampler passes processing_class.model_input_names[0] to
        #      LengthGroupedSampler. Qwen3.6's processing_class is a Processor, so that is
        #      "pixel_values", not "input_ids" -> ValueError before step 0.
        #   2. Supplying a precomputed "length" column does not help: _remove_unused_columns
        #      prunes every column outside the model forward signature, and it runs BEFORE
        #      sampler_fn in _get_dataloader, so the column is gone by sampling time.
        # Default random sampler is used instead; batches still pad to their own longest
        # member, just without length bucketing.
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
    ),
    callbacks=[PeriodicMemoryCleanup(CLEANUP_STEPS)],
)

# ===================== TRAIN ON RESPONSES ONLY =====================
# Mask prompt tokens to -100 so loss is computed ONLY on assistant turns.
# Without this the loss averages in the persona system prompt and the user turns -
# tokens the base model already predicts near-perfectly and that repeat across all
# 4,352 examples. They pin the average near their own ~0 loss and bury the signal
# from the response tokens. That is why the previous run started at only 2.23 and
# flattened around 0.9, while the masked pubmed medgemma SFT started at 7.32 and
# fell to 0.61.
#
# Both parts are left as None so Unsloth auto-detects them from the Qwen chat
# template and prints what it found - no hand-written marker strings to get wrong.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

# Verify the markers actually matched. If they do not, every label can end up -100,
# which trains on nothing and silently burns the entire run.
import numpy as np

_probe = trainer.train_dataset[:8]["labels"]
_kept = sum(int((np.array(x) != -100).sum()) for x in _probe)
_total = sum(len(x) for x in _probe)
if _kept == 0:
    raise RuntimeError(
        "train_on_responses_only masked EVERY token - the instruction/response "
        "markers did not match the chat template. Do not start training."
    )
print(f"Response masking OK: {_kept:,}/{_total:,} label tokens kept "
      f"({100 * _kept / _total:.1f}%) across 8 sample sequences")

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
print(f"Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: disabled (hybrid linear-attention model)")
print(f"  Length grouping: disabled (Processor exposes no input_ids model_input_name)")
print(f"  Checkpoints: every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Memory cleanup: every {CLEANUP_STEPS} steps")
print(f"  Loss computed on: assistant responses only (prompt masked to -100)")
print(f"  Dataset: {len(dataset)} examples")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/4352 [00:00<?, ? examples/s]

Unsloth: Auto-detected instruction_part = '\n<|im_start|>user\n' and response_part = '\n<|im_start|>assistant\n'


Map:   0%|          | 0/4352 [00:00<?, ? examples/s]

Response masking OK: 7,103/10,998 label tokens kept (64.6%) across 8 sample sequences
Trainer configured
  Effective batch size: 2 x 4 = 8
  Epochs: 1
  LR: 0.0002
  Packing: disabled (hybrid linear-attention model)
  Length grouping: disabled (Processor exposes no input_ids model_input_name)
  Checkpoints: every 50 steps, keep 3
  Memory cleanup: every 50 steps
  Loss computed on: assistant responses only (prompt masked to -100)
  Dataset: 4352 examples
  Precision: bf16


## 9. Train

In [ ]:
# Start training. Auto-resumes from the newest checkpoint in OUTPUT_DIR_ADAPTERS if one
# exists, so an interrupted run continues instead of restarting from step 0.
import os
from transformers.trainer_utils import get_last_checkpoint

_ckpt_dir = trainer.args.output_dir
last_checkpoint = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nTraining complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None}.


No checkpoint found - starting from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,352 | Num Epochs = 1 | Total steps = 544
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 159,383,552 of 27,516,112,112 (0.58% trained)


Step,Training Loss
5,1.584089
10,1.435480
15,1.176434
20,1.238906
25,1.064045
30,1.426335
35,1.321028
40,1.263596
45,1.300227
50,1.099622


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_6_27b_unsloth_4bit/train/checkpoint-50/tokenizer_config.json.


## 10. Save LoRA Adapters

Save the trained LoRA adapters plus the persona system prompts. The adapters load on any quantization of the same Qwen3.6 27B base via PEFT or vLLM.

In [ ]:
from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

print(f"\nLoRA adapters saved")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["amos"]  # or any persona key')

## 11. Test Inference

Quick smoke test with a few personas using their extracted system prompts. Each persona should respond in its distinctive voice.

Uses `ENABLE_THINKING` from Cell 1. With it `True`, output begins inside a `<think>` block before the answer, so `max_new_tokens` is raised to leave room for both.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Pick up to 4 personas to test
test_personas = list(persona_system_prompts.keys())[:4]

print(f"INFERENCE TEST — {len(test_personas)} PERSONAS (thinking={'ON' if ENABLE_THINKING else 'OFF'})\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    # Qwen3.6's tokenizer may be a multimodal Processor; calling it positionally
    # binds the string to its first param (`images`), not `text`, so pass by keyword.
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  PERSONA: {persona_key.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024 if ENABLE_THINKING else 256,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

## 12. Verify Adapter (Reload from Disk)

Final validation: load the adapter cold from disk to confirm it is self-contained and portable.

In [ ]:
# Clean up training model
import gc, torch
del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

# Reload from disk
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

# Reload saved system prompts
import json
from pathlib import Path
with open(f"{LORA_OUTPUT_DIR}/persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

# Test with first persona
test_key = list(reloaded_prompts.keys())[0]
test_prompt_text = reloaded_prompts[test_key]

messages = [
    {"role": "system", "content": test_prompt_text},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
inputs = tokenizer2(text=text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=1024 if ENABLE_THINKING else 256,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    top_k=GEN_TOP_K,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk. Ready for deployment via vLLM.")

# List adapter files
print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Reload adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model2, tokenizer2
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/gguf"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter merges before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Pass ALL quant methods in one call so the LoRA->FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge -> {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\nGGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(Path(GGUF_OUTPUT_DIR).glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nGGUF deployment (llama.cpp / Ollama / LM Studio):")
print("  1. Deploy the q4_k_m .gguf file to your target platform.")
print("  2. Load in llama.cpp-compatible tooling (Ollama, LM Studio, llamafile, etc.).")
print("  3. Configure the Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()